# Virtual products

<div align="center">
<img src="https://github.com/SciQLop/SciQLop/raw/main/SciQLop/resources/icons/SciQLop.png"/>
</div>

A **virtual product** is a Python callback registered in the product tree. SciQLop calls it on demand with `(start, stop)` and your function returns time-series data — exactly like a regular data product, but computed on the fly.

**What you'll learn**
- How to define a virtual product from a notebook using the `%%vp` cell magic.
- How to choose its return type (Scalar / Vector / MultiComponent / Spectrogram).
- How to fold a physics formula — the mirror-mode instability threshold — into a virtual product so SciQLop computes and plots it for any time interval you pan to.
- How to do the same thing programmatically with `create_virtual_product` (useful for plugins).

**Prerequisites** — [Plot panels](5-SciQLopBasicPlotPanel.ipynb), [IPython magics](7-SciQLopMagics.ipynb), [Speasy data manipulation](../Speasy/4-SpeasyDataManipulation.ipynb).

**Next** — [Parameterised virtual products (knobs)](15-SciQLopParameterizedVirtualProducts.ipynb), [DSP](14-SciQLopDSP.ipynb).


## A worked example: the mirror-mode instability threshold

In the [Speasy data manipulation](../Speasy/4-SpeasyDataManipulation.ipynb) tutorial we computed the mirror-mode threshold as a one-off scalar series. Here we wrap that exact computation in a virtual product so SciQLop recomputes it whenever you pan or zoom — no manual cell re-execution.

As a reminder, the threshold is

$$ C = \beta_\perp\left(\frac{T_\perp}{T_{\parallel}}-1\right), \qquad \beta_\perp = \frac{2\mu_0 P_\perp}{B^2} $$

so we need T‖, T⊥, n, and B from MMS1.


### The `%%vp` cell magic — the everyday way

`%%vp --path "<path>"` registers the function defined in the cell as a virtual product under `<path>` in the product tree. The return type annotation (`Scalar`, `Vector`, `MultiComponent`, `Spectrogram`) tells SciQLop how to render it; those names are auto-imported in `%%vp` cells.

Re-executing the cell hot-reloads the callback — any open graph using it picks up the new function on the next data request.


In [ ]:
%%vp --path "mms/mirror" --start "2015-11-19T13:00" --stop "2015-11-19T15:00"
import speasy as spz
import scipy.constants as cst
from speasy.signal.resampling import interpolate


def mirror_mode_threshold(start: float, stop: float) -> Scalar["Mirror mode threshold"]:
    mms1 = spz.inventories.data_tree.cda.MMS.MMS1
    products = [
        mms1.DIS.MMS1_FPI_FAST_L2_DIS_MOMS.mms1_dis_temppara_fast,
        mms1.DIS.MMS1_FPI_FAST_L2_DIS_MOMS.mms1_dis_tempperp_fast,
        mms1.FGM.MMS1_FGM_SRVY_L2.mms1_fgm_b_gse_srvy_l2,
        mms1.DIS.MMS1_FPI_FAST_L2_DIS_MOMS.mms1_dis_numberdensity_fast,
    ]
    tpara, tperp, b, n = spz.get_data(products, start, stop)
    if any(v is None for v in (tpara, tperp, b, n)):
        return None

    b = interpolate(tperp, b)
    Pperp = tperp * n * 1e6
    beta_perp = 2 * cst.mu_0 * cst.e * Pperp / (b["Bt"] * 1e-9) ** 2
    return beta_perp * (tperp / tpara - 1)


In [ ]:
from SciQLop.user_api.plot import create_plot_panel
from SciQLop.user_api import TimeRange

panel = create_plot_panel()
panel.time_range = TimeRange("2015-11-19T13:00", "2015-11-19T15:00")
panel.plot("mms/mirror")


Find **mms/mirror** in the product tree and drag it onto an existing FGM panel to overlay it next to the magnetic field. Pan the time range — SciQLop recomputes the threshold over the new interval automatically.

The `--start` / `--stop` flags above are a convenience: they tell `%%vp` to make a debug plot of the product over that range when the cell executes, so you can verify the computation works before going hunting in the GUI.

---

## The programmatic API: `create_virtual_product`

For SciQLop plugins (where there's no IPython shell) you can register the same virtual product from regular Python with `create_virtual_product`. This is the underlying API that `%%vp` calls into; the two are interchangeable.


In [ ]:
from SciQLop.user_api.virtual_products import create_virtual_product, VirtualProductType
import speasy as spz
import scipy.constants as cst
from speasy.signal.resampling import interpolate


def mirror_mode_threshold_prog(start: float, stop: float):
    mms1 = spz.inventories.data_tree.cda.MMS.MMS1
    products = [
        mms1.DIS.MMS1_FPI_FAST_L2_DIS_MOMS.mms1_dis_temppara_fast,
        mms1.DIS.MMS1_FPI_FAST_L2_DIS_MOMS.mms1_dis_tempperp_fast,
        mms1.FGM.MMS1_FGM_SRVY_L2.mms1_fgm_b_gse_srvy_l2,
        mms1.DIS.MMS1_FPI_FAST_L2_DIS_MOMS.mms1_dis_numberdensity_fast,
    ]
    tpara, tperp, b, n = spz.get_data(products, start, stop)
    if any(v is None for v in (tpara, tperp, b, n)):
        return None
    b = interpolate(tperp, b)
    Pperp = tperp * n * 1e6
    beta_perp = 2 * cst.mu_0 * cst.e * Pperp / (b["Bt"] * 1e-9) ** 2
    return beta_perp * (tperp / tpara - 1)


vp = create_virtual_product(
    "/mms/mirror_prog",
    mirror_mode_threshold_prog,
    VirtualProductType.Scalar,
    labels=["Mirror mode threshold (programmatic)"],
)
